# Phase 1: Target Definition & Baseline
## Step 1.1: 신제품(NPD) 식별 및 출시 주차별 누적 매출 분포 분석

- **목표**: 각 상품의 최초 출시일을 정의하고, 출시 후 1, 2, 4주 차의 누적 매출액 분포를 통해 '성공'의 임계값($y$)을 도출합니다.
- **데이터**: `B2_POS_SALE.parquet` (POS 판매 데이터)

In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import sys

: 

In [ ]:

# 데이터 경로 설정
B2_PATH = '../../data/processed/B2_POS_SALE.parquet'

# 1. 데이터 로드 및 컬럼명 정규화
def load_and_preprocess_b2(path):
    # 스키마 확인을 통한 컬럼 매핑 (인코딩 문제 해결)
    schema_names = pl.scan_parquet(path).collect_schema().names()
    mapping = {
        schema_names[0]: 'SALE_DATE',
        schema_names[1]: 'SALE_TIME',
        schema_names[2]: 'STORE_CODE',
        schema_names[3]: 'POS_NO',
        schema_names[4]: 'TRADE_NO',
        schema_names[5]: 'ITEM_CODE',
        schema_names[6]: 'SALE_QTY',
        schema_names[7]: 'SALE_AMT'
    }
    
    return pl.scan_parquet(path).rename(mapping)

b2_lazy = load_and_preprocess_b2(B2_PATH)

# 2. 출시일(최초 판매일) 정의
b2_with_date = b2_lazy.with_columns(
    pl.col('SALE_DATE').str.to_date('%Y%m%d').alias('sale_dt')
)

launch_dates = b2_with_date.group_by('ITEM_CODE').agg(
    pl.col('sale_dt').min().alias('launch_dt')
)

# 3. 출시 후 경과 일수 및 누적 매출 계산
sales_enriched = b2_with_date.join(launch_dates, on='ITEM_CODE')
sales_enriched = sales_enriched.with_columns(
    (pl.col('sale_dt') - pl.col('launch_dt')).dt.total_days().alias('days_after')
)

cum_sales = sales_enriched.group_by('ITEM_CODE').agg([
    pl.col('SALE_AMT').filter(pl.col('days_after') <= 7).sum().alias('cum_1w'),
    pl.col('SALE_AMT').filter(pl.col('days_after') <= 14).sum().alias('cum_2w'),
    pl.col('SALE_AMT').filter(pl.col('days_after') <= 28).sum().alias('cum_4w')
]).collect()

print(f"Total Unique Items: {len(cum_sales):,}")

In [ ]:
# 4. 시각화: 누적 매출 분포 (Log Scale)
fig = go.Figure()

for period, name in zip(['cum_1w', 'cum_2w', 'cum_4w'], ['1 Week', '2 Weeks', '4 Weeks']):
    data = cum_sales.filter(pl.col(period) > 0)[period].to_numpy()
    fig.add_trace(go.Box(y=data, name=name, boxmean='sd'))

fig.update_layout(
    title="신제품 출시 기간별 누적 매출액 분포 (Log Scale)",
    yaxis_type="log",
    yaxis_title="누적 매출액 (원)",
    template="plotly_white"
)
fig.show()

In [ ]:
# 5. 임계값(Threshold) 산출
print("--- NPD Sales Distribution Summary ---")
for period in ['cum_1w', 'cum_2w', 'cum_4w']:
    active_sales = cum_sales.filter(pl.col(period) > 0)[period]
    q80 = active_sales.quantile(0.8)
    q90 = active_sales.quantile(0.9)
    q95 = active_sales.quantile(0.95)
    print(f"{period} | Top 20% (Q80): {int(q80):,}, Top 10% (Q90): {int(q90):,}, Top 5% (Q95): {int(q95):,}")